In [34]:
!nvidia-smi

Wed May 27 19:14:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   66C    P8             18W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [35]:
!pip install --upgrade pip
!pip install wandb hugginface_hub uv
!uv pip install -U vllm --pre \
  --extra-index-url https://wheels.vllm.ai/nightly/cu129 \
  --extra-index-url https://download.pytorch.org/whl/cu129 \
  --index-strategy unsafe-best-match

ERROR: Could not find a version that satisfies the requirement hugginface_hub (from versions: none)
ERROR: No matching distribution found for hugginface_hub
Using Python 3.12.13 environment at: /usr
Resolved 185 packages in 2.45s
Prepared 4 packages in 0.60ms
Uninstalled 4 packages in 4ms
Installed 4 packages in 4ms
 - dill==0.3.8
 + dill==0.4.1
 - fsspec==2025.3.0
 + fsspec==2026.4.0
 - httpx-sse==0.4.0
 + httpx-sse==0.4.3
 - protobuf==5.29.6
 + protobuf==6.33.6


In [36]:
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_DIR = '/content/drive/MyDrive/resilient_results'
os.makedirs(RESULTS_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
from huggingface_hub import login
import os
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
wandb_api = userdata.get("WANDB_API")

login(hf_token)  # paste your HF token

import wandb
wandb.login(key=wandb_api)  # paste your W&B token

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [38]:
import os
import time
import requests # For checking server status
import subprocess

# 1. Kill any existing vLLM processes to ensure a clean start
print("Killing any existing vLLM server processes...")
# Use 'pkill -f' to find and terminate processes by command line
subprocess.run(["pkill", "-f", "vllm serve"], check=False)
time.sleep(2) # Give a moment for processes to terminate

# 2. Disable FlashInfer JIT sampling (requires nvcc which is not available on this runtime)
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

# Start vLLM server in the background using nohup and &
# Redirect stdout and stderr to a log file for debugging
vllm_log_file = "vllm_server.log"
vllm_command = f"""
nohup vllm serve google/gemma-4-E4B-it \
  --max-model-len 8192 \
  --gpu-memory-utilization 0.90 \
  --dtype float16 \
  --trust-remote-code \
  --port 8000 \
  > {vllm_log_file} 2>&1 &
"""
print(f"Starting vLLM server (output in {vllm_log_file})...")
# Execute the command directly in the shell
!{vllm_command}

# 3. Implement a robust waiting mechanism
# Check the log file for startup message or attempt a connection
server_ready = False
max_wait_time = 500 # Maximum wait time in seconds (5 minutes)
start_time = time.time()
print("Waiting for vLLM server to become ready (checking log and port)...")

while not server_ready and (time.time() - start_time) < max_wait_time:
    # Check log file for "Application startup complete"
    if os.path.exists(vllm_log_file):
        with open(vllm_log_file, "r") as f:
            log_content = f.read()
            if "Application startup complete" in log_content:
                print("Found 'Application startup complete' in log file.")
                # The log might show startup complete before the API is fully responsive,
                # so we continue to check the API endpoint.
                server_ready = True # Tentatively mark as ready from logs

    # Also try to hit a known API endpoint to confirm readiness
    try:
        # Check /v1/models endpoint, which typically lists available models
        response = requests.get("http://localhost:8000/v1/models", timeout=5)
        if response.status_code == 200:
            print("Successfully connected to vLLM server's /v1/models endpoint.")
            server_ready = True
            break # Exit loop if connection successful
    except requests.exceptions.ConnectionError:
        pass # Server not yet ready to accept connections
    except requests.exceptions.Timeout:
        pass # Connection timed out, still not ready
    except Exception as e:
        # Catch other potential request errors
        print(f"Warning: Error checking vLLM server status: {e}")

    if not server_ready:
        time.sleep(5) # Wait 5 seconds before checking again

if server_ready:
    print("✅ vLLM server is now ready.")
    # Optionally, print the last few lines of the log for confirmation
    if os.path.exists(vllm_log_file):
        with open(vllm_log_file, "r") as f:
            lines = f.readlines()
            print("--- Last 10 lines of vLLM Server Log ---")
            for line in lines[-10:]:
                print(line, end="")
            print("---------------------------------------")
else:
    print("❌ vLLM server did not become ready within the allotted time.")
    if os.path.exists(vllm_log_file):
        with open(vllm_log_file, "r") as f:
            print("--- Full vLLM Server Log ---")
            print(f.read())
            print("----------------------------")
    # If the server didn't start, we might want to kill it again just in case
    subprocess.run(["pkill", "-f", "vllm serve"], check=False)


Killing any existing vLLM server processes...
Starting vLLM server (output in vllm_server.log)...
Waiting for vLLM server to become ready (checking log and port)...
Found 'Application startup complete' in log file.
Successfully connected to vLLM server's /v1/models endpoint.
✅ vLLM server is now ready.
--- Last 10 lines of vLLM Server Log ---
(APIServer pid=23961) INFO 05-27 19:17:35 [launcher.py:46] Route: /inference/v1/generate, Methods: POST
(APIServer pid=23961) INFO 05-27 19:17:35 [launcher.py:46] Route: /scale_elastic_ep, Methods: POST
(APIServer pid=23961) INFO 05-27 19:17:35 [launcher.py:46] Route: /is_scaling_elastic_ep, Methods: POST
(APIServer pid=23961) INFO 05-27 19:17:35 [launcher.py:46] Route: /generative_scoring, Methods: POST
(APIServer pid=23961) INFO 05-27 19:17:35 [launcher.py:46] Route: /v1/chat/completions/render, Methods: POST
(APIServer pid=23961) INFO 05-27 19:17:35 [launcher.py:46] Route: /v1/completions/render, Methods: POST
(APIServer pid=23961) INFO:     St

In [39]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY"
)

response = client.chat.completions.create(
    model="google/gemma-4-E4B-it",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {"url": "https://headsupfortails.com/cdn/shop/articles/Welcoming_a_Cat_to_a_New_Home_x630.jpg"}
                },
                {
                    "type": "text",
                    "text": "Describe this image in detail."
                }
            ]
        }
    ],
    max_tokens=1024
)

print(response.choices[0].message.content)

This is a close-up photograph of a beautiful, fluffy cat with striking blue eyes.

**The Cat:**
* **Appearance:** The cat has a medium to long, thick coat, suggesting it might be a breed with a substantial amount of fluff. The coloring is a mix of gray, brown, and white, giving it a classic tabby pattern, especially noticeable on its face and body. There are prominent stripes and patches of darker fur against a lighter gray/brown base.
* **Face and Expression:** The cat is looking directly at the camera with an intense and gentle gaze. Its eyes are a vibrant, clear blue, which contrasts beautifully with its fur colors. Its facial expression appears calm, curious, and sweet. Its whiskers are long, numerous, and white, fanning out widely.
* **Pose:** The cat is lying down, resting its body on a wooden surface. Its front paws are visible in the foreground—the left paw appears to be tucked neatly, while the right paw is extended slightly forward.

**Setting and Lighting:**
* **Foreground/S

In [40]:
!git clone https://github.com/EvolvingLMMs-Lab/lmms-eval.git
!cd lmms-eval && uv pip install -e ".[all]"

fatal: destination path 'lmms-eval' already exists and is not an empty directory.
Using Python 3.12.13 environment at: /usr
Resolved 270 packages in 294ms
Prepared 1 package in 1.62s
Uninstalled 5 packages in 5ms
Installed 5 packages in 2ms
 - dill==0.4.1
 + dill==0.3.8
 - fsspec==2026.4.0
 + fsspec==2025.3.0
 - httpx-sse==0.4.3
 + httpx-sse==0.4.0
 ~ lmms-eval==0.7.1 (from file:///content/lmms-eval)
 - protobuf==6.33.6
 + protobuf==5.29.6


In [41]:
!pip install decord
!python -m lmms_eval --tasks list

2026-05-27 19:18:11 | INFO     | lmms_eval.__main__:cli_evaluate:475 - Verbosity set to INFO
2026-05-27 19:18:15 | INFO     | lmms_eval.__main__:cli_evaluate_single:594 - Evaluation tracker args: {}
2026-05-27 19:18:15 | INFO     | lmms_eval.__main__:cli_evaluate_single:640 - Available Tasks:
 - 3dsrbench
 - 3dsrbench_circular
 - ConBench
 - FALCONBench_mcq
 - FALCONBench_mcq_temploc
 - FALCONBench_oq
 - FALCONBench_oq_temploc
 - JumpScore
 - VisualPuzzles_cot
 - VisualPuzzles_direct
 - WISE
 - abench_dev
 - activitynetqa
 - ai2_arc
 - ai2d
 - ai2d_lite
 - ai2d_no_mask
 - ai2d_reasoning
 - aime24_agg8_reasoning
 - aime24_figures
 - aime24_figures_agg64
 - aime24_nofigures
 - aime24_nofigures_agg64
 - aime24_nofigures_agg8
 - aime25_agg8_reasoning
 - aime25_nofigures
 - aime25_nofigures_agg64
 - aime25_nofigures_agg8
 - aime_2024_agg8
 - aime_2024_rebase
 - aime_figures
 - aime_nofigures
 - aime_reasoning
 - air_bench_chat
 - air_bench_chat_mixed
 - air_bench_chat_music
 - air_bench_cha

In [43]:
!OPENAI_API_BASE="http://localhost:8000/v1" OPENAI_API_KEY="EMPTY" \
python -m lmms_eval \
  --model async_openai \
  --model_args model_version=google/gemma-4-E4B-it,base_url=http://localhost:8000/v1,api_key=dummy,num_cpus=4,timeout=600,is_qwen3_vl=False \
  --tasks mmmu_pro_vision \
  --force_simple \
  --process_with_media \
  --batch_size 1 \
  --log_samples \
  --wandb_log_samples \
  --output_path ./results_mmmu_pro_vision \
  --wandb_args project=saesha-parekhcivicdatalab-Gemma-4-Compression,name=mmmu_pro_vision \
  --verbosity DEBUG

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: saesha-parekh (saesha-parekhcivicdatalab) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /content/wandb/run-20260527_200016-i0o9n1pf
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run mmmu_pro_vision
wandb: ⭐️ View project at https://wandb.ai/saesha-parekhcivicdatalab/saesha-parekhcivicdatalab-Gemma-4-Compression
wandb: 🚀 View run at https://wandb.ai/saesha-parekhcivicdatalab/saesha-parekhcivicdatalab-Gemma-4-Compression/runs/i0o9n1pf
2026-05-27 20:00:18 | INFO     | lmms_eval.__main__:cli_evaluate:475 - Verbosity set to DEBUG
2026-05-27 20:00:18 | DEBUG    | lmms_eval.tasks:_get_task_and_group:484 - File _default_template.yaml in /content/lmm